In [1]:
import pathlib

import yaml
import pandas as pd

import jax.random as jr

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

In [6]:
# input_files_stem = pathlib.Path.cwd().parent / 'kkkkk'
# input_files_stem = pathlib.Path.cwd().parent / 'z5'
# input_files_stem = pathlib.Path('/tmp') / 'z3'
# input_files_stem = pathlib.Path('/tmp') / 'z5'
# input_files_stem = pathlib.Path('/tmp') / 'z10'
# input_files_stem = pathlib.Path('/tmp') / 'z20'
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'

In [7]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [8]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
parameters

{'PRNG seeds': {'model initialization': {'decoder': 43, 'encoder': 42},
  'sampling': {'training set': 14, 'validation set': 15},
  'training dataloader': 43,
  'training-validation splitting': 42},
 'encoder': {'input_size': [1, 10000]},
 'loss': {'x_sd': 0.5},
 'model': {'channels': [12, 12, 12],
  'dim_z': 10,
  'kernel_sizes': [128, 64, 32],
  'stride': 2},
 'training': {'batch_size': 32,
  'dl_rng_key': 43,
  'learning_rate': 3e-05,
  'n_epochs': 200,
  'train_test_split': {'test_size': 0.2}}}

In [9]:
results_df = pd.read_parquet(results_file)
results_df

training                             validation                \
             loss loglikelihood         kl          loss loglikelihood   
0    21141.457260 -21115.471004  25.986215  17962.017685 -17918.889827   
1    16665.591487 -16614.024462  51.567047  15633.292742 -15574.613103   
2    15347.333269 -15285.323194  62.010092  14887.008360 -14824.312047   
3    14798.445325 -14735.558356  62.886943  14490.936369 -14427.941918   
4    14467.642545 -14405.342272  62.300302  14242.389815 -14180.279785   
..            ...           ...        ...           ...           ...   
195   9870.216385  -9816.227464  53.988911  11513.190751 -11460.529309   
196   9866.314100  -9812.432037  53.882075  11475.487793 -11421.291814   
197   9861.418712  -9807.275556  54.143146  11479.430831 -11426.003430   
198   9852.927725  -9798.793211  54.134553  11501.499524 -11448.628978   
199   9852.204459  -9798.188655  54.015775  11557.977384 -11504.299209   

                
            kl  
0    43.127894  
1    58.679642  
2    62.696297  
3    62.994401  
4    62.110062  
..         ...  
195  52.661473  
196  54.195963  
197  53.427433  
198  52.870559  
199  53.678188  

[200 rows x 6 columns]

In [10]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [11]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [24]:
sum(trapallada.util.get_layers_n_parameters(model).values())

861141

In [12]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

In [13]:
# batch = next(iter(train_dl))
batch = next(iter(validation_dl))

In [14]:
ds.metadata_df

,label,SNR,color,file name
0,FM,3.162,None,FM/FM_SNR3.162_Simul53.mat
1,FM,10.000,None,FM/FM_SNR10_Simul450.mat
2,FM,3.162,None,FM/FM_SNR3.162_Simul245.mat
3,FM,31.620,None,FM/FM_SNR31.62_Simul30.mat
4,FM,31.620,None,FM/FM_SNR31.62_Simul947.mat
...,...,...,...,...
12978,noise,NaN,1.6987,noise/noise_color1.6987.mat
12979,noise,NaN,1.9692,noise/noise_color1.9692.mat
12980,noise,NaN,0.19187,noise/noise_color0.19187.mat
12981,noise,NaN,1.3106,noise/noise_color1.3106.mat


In [18]:
i = 0
# i = 25

i_s, x_s, y_s = batch
x_s = x_s.numpy()
x_pred, z_mean, z_sd = model(x_s[i], jr.PRNGKey(42))

display(ds.metadata_df.loc[i_s[i].item()])

df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_s[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

label                                           carrier
SNR                                               3.162
color                                              None
file name    singlePeriod/carrier_SNR3.162_Simul931.mat
Name: 8940, dtype: object

alt.Chart(...)

In [16]:
z_mean, z_sd

(Array([ 0.68607706, -1.2203076 ,  0.6761613 ,  0.46356112,  0.35420036,
        -2.1511095 , -2.5505495 ,  1.1889175 ,  1.1500769 ,  0.31341672],      dtype=float32),
 Array([0.01012006, 0.01756719, 0.01109941, 0.01198461, 0.00708726,
        0.01630088, 0.0185939 , 0.0224893 , 0.01432728, 0.00976618],      dtype=float32))

In [17]:
model.encoder(x_s[0])

Array([-0.45212248, -2.0546746 , -1.7083219 ,  3.731602  , -3.1851208 ,
       -2.5863876 , -1.0733359 ,  1.673935  ,  0.0339    , -1.8369924 ,
        0.01913868,  0.01664177,  0.01299126,  0.03122932,  0.01603218,
        0.01228553,  0.01618055,  0.01450766,  0.02083127,  0.02017381],      dtype=float32)